### Extract and Prepare CSV data from YFinance  
- Import essential packages including yfinance and pandas.
- import extract_data.py module and call many_tickers function to perform:
    - Download historical stock price data for major tech companies (AAPL, MSFT, etc.) from Yahoo Finance using the yfinance library. 
    - Fetch daily prices between January 2020 and June 2025, then processes the multi-level column structure into a clean, long-format DataFrame. 
    - Create a 'data' directory and saves the final output as a CSV file with columns for Date, Ticker, and various price metrics (Open, High, Low, Close, etc.), making it ready for analysis. 
- Transform data as CSV.

In [1]:
import yfinance as yf
import pandas as pd
import os
import datetime
import extract_data
import pandas as pd

os.makedirs('data', exist_ok=True)
tickers = extract_data.many_tickers(['AAPL', 'MSFT', 'GOOG', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA'], '2020-01-01', '2025-06-30')
tickers.to_csv('data/combined_tickers.csv', index=False)
print("Saved combined data to yfinance_data/combined_tickers.csv")


/home/joshlai/NTU-Project-Data-Science-AI/extract_data.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(tickers, start=start_date, end=end_date)
[*********************100%***********************]  8 of 8 completed


Saved combined data to yfinance_data/combined_tickers.csv


### Extract and Prepare CSV data from FRED

- Call many_fred_data function from extract_data.py module to perform:
    - Fetch key U.S. economic indicators from the **FRED (Federal Reserve Economic Data)** database (from January 1, 2020 to June 1, 2025).  
    - Combine all metrics into a single **DataFrame**.  
- Export to CSV.

In [2]:
from pandas_datareader import data as pdr

econ_data = extract_data.many_fred_data(['CPIAUCSL', 'FEDFUNDS', 'UNRATE','GDP'], ['CPI', 'Federal_Funds_Rate', 'Unemployment_Rate','GDP'], 'fred', '2020-01-01', '2025-06-30')

# Show data
print(econ_data.head())

# Save to CSV
econ_data.to_csv('data/economic_data_fred.csv')


        DATE      CPI  Federal_Funds_Rate  Unemployment_Rate        GDP
0 2020-01-01  259.127                1.55                3.6  21727.657
1 2020-02-01  259.250                1.58                3.5        NaN
2 2020-03-01  258.076                0.65                4.4        NaN
3 2020-04-01  256.032                0.05               14.8  19935.444
4 2020-05-01  255.802                0.05               13.2        NaN


### Transform (Merge and Standardize) Data 
  - Combine datasets using a **left join** on the date field, unifying column names (`DATE` → `date`).  
  - Drop duplicate date columns and **renames all columns to lowercase** for consistency.  
  - Add a `stock_price_id` for unique identification.  
  - Reorder columns logically, prioritizing ID, date, ticker, and market/economic variables. 

In [3]:
# Load CSVs
stock_df = pd.read_csv('data/combined_tickers.csv', parse_dates=['Date'])
econ_df = pd.read_csv('data/economic_data_fred.csv', parse_dates=['DATE'])

# Merge and immediately rename date to a unified name
merged_df = pd.merge(
    stock_df,
    econ_df,
    left_on='Date',
    right_on='DATE',
    how='left'
)

# Drop duplicate date column and rename
merged_df = merged_df.drop(columns=['DATE']).rename(columns={'Date': 'date'})

# Optional: add surrogate key
merged_df['stock_price_id'] = merged_df.index + 1

# Reorder columns using lowercase, standardized names
merged_df = merged_df.rename(columns={
    'Ticker': 'ticker',
    'Open': 'open',
    'High': 'high',
    'Low': 'low',
    'Close': 'close',
    'Volume': 'volume',
    'CPI': 'cpi_value',
    'Federal_Funds_Rate': 'interest_rate'
})

# Final column arrangement
fact_stock_prices = merged_df[
    ['stock_price_id', 'date', 'ticker', 'open', 'high', 'low', 'close', 'volume', 'cpi_value', 'interest_rate']
]

# Preview
print(fact_stock_prices.head())


   stock_price_id       date ticker        open        high         low  \
0               1 2020-01-02   AAPL   71.627062   72.681258   71.373188   
1               2 2020-01-02   AMZN   93.750000   94.900497   93.207497   
2               3 2020-01-02   GOOG   66.681136   68.002778   66.681136   
3               4 2020-01-02  GOOGL   67.018569   68.026024   66.923141   
4               5 2020-01-02   META  205.780159  208.805892  205.302415   

        close       volume  cpi_value  interest_rate  
0   72.620811  135480400.0        NaN            NaN  
1   94.900497   80580000.0        NaN            NaN  
2   67.964508   28132000.0        NaN            NaN  
3   68.026024   27278000.0        NaN            NaN  
4  208.795944   12077100.0        NaN            NaN  


### Creates a time dimension table (`dim_date`) for data warehousing/analysis
  - Covers every day from **January 1, 2020**, to **December 31, 2025**.  
  - Basic: `year`, `month`, `day`.  
  - Temporal: `weekday` (name), `quarter` (1-4).  
  - Each row represents a **unique date** with derived temporal features.

In [4]:
# Define the range of dates
date_range = pd.date_range(start='2020-01-01', end='2025-12-31')

# Create the dimension table
dim_date = pd.DataFrame({
    'date': date_range,
    'year': date_range.year,
    'month': date_range.month,
    'day': date_range.day,
    'weekday': date_range.day_name(),
    'quarter': date_range.quarter
})

# Show preview
print(dim_date.head())


        date  year  month  day    weekday  quarter
0 2020-01-01  2020      1    1  Wednesday        1
1 2020-01-02  2020      1    2   Thursday        1
2 2020-01-03  2020      1    3     Friday        1
3 2020-01-04  2020      1    4   Saturday        1
4 2020-01-05  2020      1    5     Sunday        1


### Setup DBT

Initialize a new dbt project named `stock_analysis` and configure the dbt project through the interactive terminal. 
(Note: Only run the bash command in Ubuntu CLI, not Jupyter Notebook. Jupyter Notebook cannot support interative terminal)

```bash
dbt init stock_analysis

### Interactive Terminal
Which database would you like to use?
[1] bigquery
Enter a number: 1

[1] oauth
[2] service_account
Desired authentication method option (enter a number): 1

project (GCP project id): <your_project_id>

dataset (the name of your dbt dataset): yfinance_econdata

threads (1 or more): 1

job_execution_timeout_seconds [300]:

[1] US
[2] EU
Desired location option (enter a number): 1
```

### Check DBT profile
Check `stock_analysis` profile added is into `~/.dbt/profiles.yml`. 
```bash
nano ~/.dbt/profiles.yml
```

### Check DBT project directory
We use `tree` bash command to analyse the `stock_analysis` dbt project structure.
To run `tree`, please ensure that `tree` is installed in advanced. 
```bash
sudo apt install tree 
```
Check `stock_analysis` dbt project is created. Change from the working directory to the newly created `stock_analysis` dbt project folder.

In [9]:
%%bash
cd stock_analysis
tree

.
├── README.md
├── analyses
├── dbt_project.yml
├── macros
├── models
│   └── example
│       �

��── my_first_dbt_model.sql
│       ├── my_second_dbt_model.sql
│       └── schema.yml
├── seeds
├── snapshots
└── tests

8 directories, 5 files
